In [1]:
!pip install deap==1.4.2
!pip install scoop
!pip install bokeh
!pip install numba

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
# matplotlib.patches allows us create colored patches, we can use for legends in plots
import matplotlib.patches as mpatches
# seaborn also builds on matplotlib and adds graphical features and new plot types
import requests
import seaborn as sns
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

## loading DSM from Excel file

In [2]:
import pandas as pd
dsm_file_path = "Datasets/dsm_optimize_for_consolidation_clean.xlsx"
DSM = pd.read_excel(dsm_file_path, sheet_name="dsm", index_col=0)
grouping = pd.read_excel(dsm_file_path, sheet_name="grouping")
consolidation_candidates = pd.read_excel(dsm_file_path, sheet_name="consolidation")
DSM_header_list = DSM.columns

In [29]:
DSM

,Alex,Ben,Clara,David,Eva,Felix,Grace,Henry,Ingrid
Person,,,,,,,,,
Alex,0,1,1,0,0,0,0,0,0
Ben,1,0,1,0,0,0,0,0,0
Clara,1,1,0,0,0,0,0,0,0
David,0,0,0,0,1,1,0,0,0
Eva,0,0,0,1,0,1,0,0,0
Felix,0,0,0,1,1,0,0,0,0
Grace,0,0,0,0,0,0,0,1,1
Henry,0,0,0,0,0,0,1,0,1
Ingrid,0,0,0,0,0,0,1,1,0


In [30]:
grouping

,Unit,Person,UnitID
0,Unit 1,Alex,0
1,Unit 1,Ben,0
2,Unit 1,Clara,0
3,Unit 2,David,1
4,Unit 2,Eva,1
5,Unit 2,Felix,1
6,Unit 3,Grace,2
7,Unit 3,Henry,2
8,Unit 3,Ingrid,2


In [5]:
consolidation_candidates

,Candidate 1,Candidate 2
0,Unit 1,Unit 2


# Definition of 3 term Fitness function with consolidation

### Metric used is the Minimum Description length Principle defined in: Tian-Li Yu , Ali A. Yassine, David E. Goldberg paper - Eqation 3 - MDL clustering metric
### fDSM(M) = (1 - alpha - beta) * (Nc * log(Nn) + (log(Nn)*SUM_i_to_Nc(CLi)) + (alpha * [|S1| * (2log(Nn + 1))] + beta[|S2| * (2log(Nn + 1))])

##### Nc = number of clusters in the DSM
##### Nn = number of rows/columns in the DSM (ie. DSM elements such as number of roles in organization)
##### CLi = number of nodes in cluster i
##### S1 = Sum(Type 1 error) - Type 1 error = Every relationship which is not included in a cluster is a type 1 error
##### S2 = Sum(Type 2 error) - Type 2 error = A node is included in a cluster, but there are directional relationships with other nodes in the 
##### S3= Sum(Type 3 error) - Type 3 error = Nodes that should be consolidated, but are not
##### cluster which are not present (ie each 0 weight between two nodes is a Type 2 error)
##### logarithm base in the equation is 2. The equation is designed to operate on a binary genome. Any weights should be normalized between 0 and 1

## TODO:



(1)

 

Organizational redundancy factor = 
(no. of interations outside of clusters) / ((no. of interations outside of clusters) + (no. of interations inside clusters) ) 

 

 

Theoretically becomes, 

- 1 if all interactions are outside of clusters (maximum redundancy, means minimal organizational efficiency)

- 0 if all interactions are inside clusters (minimum redundancy, means maximum organizational efficiency)

 

For some internediate amount of redundancy the factor reaches a critical  value, where organizational communication becomes inefficient 

 

 

(2)

To avoid values with only one cluster, we can multiply this by

 

Organizational simplicity factor = (no. of clusters - 1) / (no. of clusters)

 

If only one cluster the efficiency is zero

the more clusters the closer the Organizational simplicity factor is to one (and has no effect on the organizational redundancy factor)

 

 


In [74]:
consolidation_candidates

,Candidate 1,Candidate 2
0,Unit 1,Unit 2


In [75]:
grouping

,Unit,Person,UnitID
0,Unit 1,Alex,0
1,Unit 1,Ben,0
2,Unit 1,Clara,0
3,Unit 2,David,1
4,Unit 2,Eva,1
5,Unit 2,Felix,1
6,Unit 3,Grace,2
7,Unit 3,Henry,2
8,Unit 3,Ingrid,2


In [76]:
# Translate the consolidation candidates to resource mapping for optimization
units_to_consolidate = consolidation_candidates.values.tolist()
consolidation_indexes = [grouping[grouping["Unit"].isin(pair)].index.tolist() for pair in units_to_consolidate]
consolidation_indexes

[[0, 1, 2, 3, 4, 5]]

In [77]:
def evalDSMmin(individual):
    DSM_eval_list = DSM_eval.values.T.tolist()
    resources_to_consolidate_table = consolidation_indexes
    
    #set all weights to 1
    for column in range(len(DSM_eval_list)):
        for row in range(len(DSM_eval_list)):
            if DSM_eval_list[column][row] > 1:
                DSM_eval_list[column][row] = 1
    
    
    Nn = NUMBER_OF_DSM_ELEMENTS
    Nc = 0
    S1 = 0
    S2 = 0
    S3 = 0
    CLi = [0] * MAX_NUMBER_OF_CLUSTERS
    
    #######CLi
    #Count number of elements in each cluster. 
    #Clusters do not overlap, so sum(CLi) will be equal to or les than NUMBER_OF_DSM_ELEMENTS
    for element_index in range(NUMBER_OF_DSM_ELEMENTS):
        CLi[individual[element_index]] += 1
    
    
    ########Nc
    #Count of number of clusters in individual (Number of cluster numbers used)
    for cluster in CLi:
        if cluster > 0:
            Nc += 1
    
    
    # Find type 1, type 2 and type 3 errors, then sum each to find the S1, S2 and S3 elements for the fitness function
    for column_index in range(NUMBER_OF_DSM_ELEMENTS):
        for row_index in range(NUMBER_OF_DSM_ELEMENTS):
            
            # Ignore the diagonal of the DSM matrix
            if column_index != row_index:
                # Type 1 error
                #print('COLUMN', column_index, 'row', row_index, 'VALUE', DSM_eval_list[column_index][row_index])
                # if element1 has a connection to element2, but they are not in the same cluster, a type 1 error has occured
                if DSM_eval_list[column_index][row_index] != 0 and individual[column_index] != individual[row_index]:
                    S1 += 1
                # Type 2 error
                # if element1 does NOT have a connection to element2, but they are in the same cluster, then a type 2 error has occured
                if DSM_eval_list[column_index][row_index] == 0 and individual[column_index] == individual[row_index]:
                    S2 += 1
                # Type 3 error
                # If two elements are candidates for consolidation, but are part of separate units, then a type 3 error has occured
                if individual[column_index] != individual[row_index] and any(column_index in candidate and row_index in candidate for candidate in resources_to_consolidate_table):
                    S3 += 1
    
    #Put it all together into: fitness = (1 - alpha - beta) * (Nc * log(Nn) + (log(Nn)*SUM_i_to_Nc(CLi)) + (alpha * [|S1| * (2log(Nn + 1))] + beta[|S2| * (2log(Nn + 1))])
    MDL_weight = 1 - alpha - beta - gamma
    
    #model_description_lenght = Nc * log(Nn) + (log(Nn)*SUM_i_to_Nc(CLi)
    MDL = Nc * math.log(Nn,2) + (math.log(Nn,2) * sum(CLi))
    
    #(alpha[|S1| * (2log(Nn + 1))] + beta[|S2| * (2log(Nn + 1))]) - number_of_elements_in_multiple_clusters
    log_factor = 2 * math.log(Nn + 1, 2)
    type_error_score = (
        alpha * S1 * log_factor + 
        beta * S2 * log_factor +
        gamma * S3 * log_factor
    )
    
    fitness = MDL_weight * MDL + type_error_score
    #print("Fitness: ", fitness, " S1: ", S1, " S2: ", Nc, " Cluster: ", individual)
    #print('MDL_weight', MDL_weight)
    #print('MDL', MDL)
    return [fitness]

# Optimization Algorithm

In [78]:
#Setting up Bokeh plot for sliding timeseries during evolution for logging
import numpy as np
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, push_notebook, show
from bokeh.models import HoverTool, ColumnDataSource
output_notebook()

# Set up widgets
#gen_list = Slider(title="Generation", value=0.0, start=.0, end=NUMBER_OF_GENERATOINS, step=1)
#avg = Slider(title="Average", value=0.0, start=.0, end=NUMBER_OF_GENERATOINS, step=1)
#min_ = Slider(title="Minimum", value=0.0, start=0.0, end=NUMBER_OF_GENERATOINS, step=1)
#max_ = Slider(title="Maximum", value=0.0, start=0.0, end=NUMBER_OF_GENERATOINS, step=1)

gen_list=[0]
avg=[0]
min_=[0]
max_=[0]

ys=[avg,min_,max_]
xs=[gen_list]*len(ys)
source=ColumnDataSource(dict(Generation=[], Average=[], Minimum=[], Maximum=[]))
#plot = figure(plot_width=800, plot_height=400, tools="")
plot = figure(width=800, height=400, tools="")
plot.x_range.follow = "end"
#plot.x_range.follow_interval = 1
#plot.x_range = Range1d(start=1, end=2) 
plot.x_range.range_padding = 0
plot.xaxis.axis_label = "Generation"
plot.yaxis.axis_label = "Fitness"
plot.legend.location = "top_left"
plot.line(source=source,x='Generation', y='Average', line_width=2, color="black", line_join="round")
plot.line(source=source,x='Generation', y='Minimum', line_width=2, color="red",line_join="round")
plot.line(source=source,x='Generation', y='Maximum', line_width=2, color="green",line_join="round")

Loading BokehJS ...

C:\Users\runarso\AppData\Local\Temp\ipykernel_27352\3734787802.py:30: UserWarning: 
You are attempting to set `plot.legend.location` on a plot that has zero legends added, this will have no effect.

Before legend properties can be set, you must add a Legend explicitly, or call a glyph method with a legend parameter set.

  plot.legend.location = "top_left"


GlyphRenderer(id='p1964', ...)

In [79]:
### Parameters for the evolutionary algorithm ###
DSM_eval = DSM
DSM_head = DSM_header_list 
#checkpoint_file = csv_file + "_checkpoint.pkl"
#checkpoint_logbook_file = csv_file + "_logbook.pkl"

if DSM_head[0] == ',' or DSM_head[0] == '':
    DSM_head = DSM_head[1:]
MAX_NUMBER_OF_CLUSTERS = 3
NUMBER_OF_DSM_ELEMENTS = len(DSM_eval.columns)
#print DSM_eval
print("Max number of clusters allowed: ", MAX_NUMBER_OF_CLUSTERS)
print("Number of elements in the organization: ", NUMBER_OF_DSM_ELEMENTS)
alpha = 0.3 # Within cluster
beta = 0.2 # Between cluster
gamma = 0.3 # Consolidation
genome_len = NUMBER_OF_DSM_ELEMENTS #*MAX_NUMBER_OF_CLUSTERS
POPULATION_SIZE = 200
NUMBER_OF_GENERATOINS=400
cxpb=0.5  #Crossover probability
mutpb=0.1 #Mutation probability

Max number of clusters allowed:  3
Number of elements in the organization:  9


In [ ]:
## No need to run this if not using checkpointing
import os
try:
    os.remove(checkpoint_file) # in case of resetting long running optimizations
    print("Deleted existing Checkpoint")
except FileNotFoundError as e:
    print("No checkpoint found")

NameError: name 'checkpoint_file' is not defined

### Optimization parameters

In [52]:
# DSM GA - Unweighted and ignores busses ###
import pickle
import deap as ea
import random
from random import sample
from functools import partial
from deap import creator, base, tools, algorithms
from numba import jit
from numpy import arange
from scoop import futures


ea.creator.create("FitnessMin", ea.base.Fitness, weights=(-1.0,))
ea.creator.create("Individual", list, fitness=ea.creator.FitnessMin)

toolbox = ea.base.Toolbox()
#toolbox.register("map", futures.map) #Set DEAP to run on a machine cluster using SCOOP

# An individual is defined as a list of cluster assignments. Each index matches the index of the DSM elements. 
# If the individual has index 2 set to 3 that means that element at index 2 in the DSM is assigned to cluster 3.
toolbox.register("attribute", random.randint, 0, MAX_NUMBER_OF_CLUSTERS-1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attribute, n=genome_len)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", evalDSMmin)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, indpb=0.05, low= 0, up=MAX_NUMBER_OF_CLUSTERS-1)
# Use elitism + normal ranking
toolbox.register("select", tools.selTournament, tournsize=5)

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)
stats.register("max", np.max)
logbook = tools.Logbook()

### Optimization loop

In [65]:
#Create the output graph in this cell
handle = show(plot, notebook_handle=True)
# jit decorator tells Numba to compile this function.
# The argument types will be inferred by Numba when function is called.
#@jit
def run_evolution():
    hallOfFame = tools.HallOfFame(2)
    population = toolbox.population(n=POPULATION_SIZE)
    tools.initIterate(list, partial(sample, range(MAX_NUMBER_OF_CLUSTERS), MAX_NUMBER_OF_CLUSTERS))
    
    gen = 0;
    while gen < NUMBER_OF_GENERATOINS:
        if gen==0:
            try:
                #checkpoint = pickle.load(open(checkpoint_file, "rb"))
                #population = checkpoint["population"]
                #gen = checkpoint["generation"]
                #random.setstate(checkpoint["rndstate"])
                print("Checkpoint found, starting from generation ", gen)
            except IOError:
                print("No checkpoint found, starting from scratch")
        
        # Update population
        population = toolbox.select(population, k=len(population))
        population = [toolbox.clone(ind) for ind in population]
        population = ea.algorithms.varAnd(population, toolbox, cxpb=cxpb, mutpb=mutpb, )
        
        offspring = [individual for individual in population if not individual.fitness.valid]
        fits = toolbox.map(toolbox.evaluate, offspring)
        for fit, ind in zip(fits, offspring):
            ind.fitness.values = fit
        record = stats.compile(population)
        logbook.record(gen=gen, evals=10, **record)
        
        #Update hall of fame to ensure we always know the best found solution
        hallOfFame.update(offspring)
        
        
        # Update logging output
        if gen%10 == 0:
            gen_list, avg, min_, max_ = logbook.select("gen", "avg", "min", "max")
            stream_record=dict(Average=[record["avg"]], Minimum=[record["min"]], Maximum=[record["max"]], Generation=[gen])
            # print(stream_record['Minimum'])
            source.stream(stream_record)
            push_notebook(handle=handle)
            
        # Checkpoint the evolution
        if gen%1000 == 0 and gen > 0:
            checkpoint = dict(population=population, generation=gen, rndstate=random.getstate())
            pickle.dump(checkpoint, open(checkpoint_file, "wb"), 4)
            pickle.dump(logbook, open(checkpoint_logbook_file, "wb"), 4)
            print("Checkpointed " + filename_in + " at generation ", gen)
            
        gen += 1
    return hallOfFame[0] #tools.selBest(offspring, k=1)[0]

#%timeit 

best = run_evolution()
print("Best found solution found: ", best)
print('final fitness', evalDSMmin(best))

Checkpoint found, starting from generation  0
Best found solution found:  [0, 0, 0, 0, 0, 0, 1, 1, 1]
final fitness [30.8917172863621]


# Export optimized model to Excel

In [27]:
print(DSM_header_list)
print(NUMBER_OF_DSM_ELEMENTS)
print(best)

Index(['Alex', 'Ben', 'Clara', 'David', 'Eva', 'Felix', 'Grace', 'Henry',
       'Ingrid'],
      dtype='object')
9
[2, 2, 2, 2, 2, 2, 1, 1, 1]


In [21]:
grouping["UnitID"] = grouping["Unit"].astype("category").cat.codes
initial_grouping = grouping["UnitID"].tolist()
initial_grouping

[0, 0, 0, 1, 1, 1, 2, 2, 2]

In [22]:
print("Fitness comparison:")
print("Optimized fitness", evalDSMmin(best)[0])
print("original fitness", evalDSMmin(initial_grouping)[0])

Fitness comparison:
Optimized fitness 30.8917172863621
original fitness 43.48464342824506


In [70]:
def style_func(df):

    styled = pd.DataFrame('', index=df.index, columns=df.columns)
    for i in range(len(df)):
        for j in range(len(df.columns)):
            val = df.iloc[i, j]
            styled.iloc[i, j] = background_color(val, i, j)
    return styled

def background_color(val, row, col):
    g1 = group_assignments[row]
    g2 = group_assignments[col]
    group_number = group_assignments[col] if g1 == g2 else -1
    rgba = cmap(norm(group_number))
    return f'background-color: {mcolors.rgb2hex(rgba)}' if group_number != -1 else ''

    
norm = mcolors.Normalize(min(group_assignments), max(group_assignments))
cmap = plt.cm.YlOrRd  # or any other like 'viridis', 'coolwarm'

group_assignments = list(best)
DSM_styled = DSM.style.apply(style_func, axis=None)
DSM_styled

,Alex,Ben,Clara,David,Eva,Felix,Grace,Henry,Ingrid
Person,,,,,,,,,
Alex,0,1,1,0,0,0,0,0,0
Ben,1,0,1,0,0,0,0,0,0
Clara,1,1,0,0,0,0,0,0,0
David,0,0,0,0,1,1,0,0,0
Eva,0,0,0,1,0,1,0,0,0
Felix,0,0,0,1,1,0,0,0,0
Grace,0,0,0,0,0,0,0,1,1
Henry,0,0,0,0,0,0,1,0,1
Ingrid,0,0,0,0,0,0,1,1,0


In [71]:
# This block doesn't run. Do we need it, this is done below.
optimized_path = dsm_file_path.replace(".xlsx", "_optimized.xlsx")
DSM_styled.to_excel(optimized_path, ) #something missing here.

In [315]:
grouping

,Unit,Person,UnitID
0,Unit 1,Alex,0
1,Unit 1,Ben,0
2,Unit 1,Clara,0
3,Unit 2,David,1
4,Unit 2,Eva,1
5,Unit 2,Felix,1
6,Unit 3,Grace,2
7,Unit 3,Henry,2
8,Unit 3,Ingrid,2


In [72]:
DSM_header_list

Index(['Alex', 'Ben', 'Clara', 'David', 'Eva', 'Felix', 'Grace', 'Henry',
       'Ingrid'],
      dtype='object')

In [56]:
df_optimized_grouping = pd.DataFrame({
    "Unit": ["Unit " + str(i) for i in best],
    "Person": DSM_header_list
})
df_optimized_grouping

,Unit,Person
0,Unit 0,Alex
1,Unit 0,Ben
2,Unit 0,Clara
3,Unit 0,David
4,Unit 0,Eva
5,Unit 0,Felix
6,Unit 2,Grace
7,Unit 2,Henry
8,Unit 2,Ingrid


In [73]:
optimized_path = dsm_file_path.replace(".xlsx", "_optimized.xlsx")
with pd.ExcelWriter(optimized_path, engine="openpyxl") as writer:
    DSM_styled.to_excel(writer, sheet_name="dsm")
    df_optimized_grouping.to_excel(writer, sheet_name="grouping")
    consolidation_candidates.to_excel(writer, sheet_name="consolidation")
